# Laboratorio 16 — Aprendizaje No Supervisado
A diferencia del aprendizaje supervisado, aquí el modelo trabaja con datos **sin etiquetas**. El objetivo es encontrar patrones ocultos, agrupaciones o estructuras en los datos. Los algoritmos principales incluyen K-Means, DBSCAN y clustering jerárquico.

## 1. Importación de librerías
Usamos `PCA` (reducción), `KMeans` (clustering), `TSNE` (visualización 2D) y `cifar10` como dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from tensorflow.keras.datasets import cifar10

## 2. Carga y preparación de datos
Cargamos CIFAR-10 (60,000 imágenes 32×32×3) y aplanamos cada imagen a un vector de 3,072 características. K-Means y PCA requieren datos tabulares (2D), no tensores 3D.

In [ ]:
# Cargar el dataset CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [ ]:
# Reorganizar los datos
X_train_flat = X_train.reshape(X_train.shape[0], -1)
print(f"dimensión de X_train despues de aplanar: {X_train_flat.shape}")

## 3. Clustering con K-Means
Aplicamos K-Means con `n_clusters=10` (10 clases de CIFAR-10). El algoritmo agrupa imágenes por similitud de píxeles **sin usar etiquetas**. Analizamos la distribución de imágenes por cluster.

In [ ]:
# Aplicar K-Means
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(X_train_flat)

In [ ]:
clusters = kmeans.labels_

In [ ]:
unique,counts = np.unique(clusters,return_counts=True)
print("Ditribución de las imágenes en los clusters:", dict(zip(unique, counts)))

## 4. Visualización de clusters
Mostramos 5 imágenes de cada cluster para evaluar cualitativamente si K-Means agrupa imágenes similares (aviones con aviones, gatos con gatos, etc.).

In [ ]:
# Visualizar los clusters
def plot_images(images, titles, rows, cols):
  fig, axes = plt.subplots(rows, cols, figsize=(15, 15))
  for i, ax in enumerate(axes.flat):
    ax.imshow(images[i])
    ax.set_title(titles[i])
    ax.axis('off')
  plt.show()


In [ ]:
sample_images = []
sample_titles = []
for i in range(10):
  idx = np.where(clusters == i)[0][:5]
  sample_images.extend(X_train[idx])
  sample_titles.extend([f"Cluster {i}"] * 5)

In [ ]:
plot_images(sample_images, sample_titles, rows=5, cols=10)

## 5. Reducción con PCA
PCA reduce 3,072 dimensiones a 50 conservando la máxima varianza. Esto acelera t-SNE y elimina ruido. PCA encuentra las direcciones donde los datos varían más.

In [ ]:
# Reducir dimensionalidad con PCA
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_train_flat)

In [ ]:
print(f"dimensión de X_train despues de aplicar PCA: {X_pca.shape}")

## 6. Visualización con t-SNE
t-SNE proyecta de 50 dimensiones a 2D para graficar. Cada punto es una imagen coloreada por el cluster asignado. Si los puntos del mismo color se agrupan, K-Means funciona bien.

In [ ]:
# Visualización con t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_pca)

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=clusters, cmap='tab10', s=10)
plt.title('Visualización de clusters con t-SNE')
plt.colorbar()
plt.show()






## Conclusión

Exploraste los fundamentos del aprendizaje no supervisado, donde el modelo descubre patrones por sí mismo sin necesidad de etiquetas. Esta técnica es fundamental para segmentación de clientes, detección de anomalías y reducción de dimensionalidad.